# Laya V4.4 — Speed-first, teacher-fidelity-gated CeNN research

**Purpose:** produce a modified copy of the verified V4.3 notebook, and independently benchmark **real bounded-window attention** against dense attention. The original native teacher, its FFNs, residuals, RoPE and layer norms are unchanged. V4.4 is **experimental**, and does not claim a speedup until new CUDA benchmarks pass.

**Why:** the uploaded V4.3 audit scored 1.0 teacher agreement / 0.845 gold accuracy / 0 gold drop, but only 0.625× native speed. Most non-native branches are nearly zero yet computed unconditionally. This notebook adds a selectively enabled local-only fast path, strict speed gating, separate output namespace, no unnecessary joint recovery, and a separate correctness-checked window kernel experiment.

**Execution:** Run cells 1–3 to create and download the modified **full V4.4 experiment notebook**; open that downloaded notebook in Colab and run it. Run cell 4 onward here for standalone kernel benchmarks. `FAST_FULL_LOCAL_ONLY=True` is an approximation and must clear teacher/gold/local gates. Native model remains the deployment baseline.

V4.4 experiment defaults run all 28 candidate evaluations. Even when an adapter does not meet the speed criterion, its attempt is logged and the original attention is kept. Strict speed gating can result in zero accepted replacements, which is a valid negative result.

In [ ]:
#@title 1. Dependencies and GPU
%pip -q install nbformat numpy
import torch, nbformat, json, pathlib, urllib.request, time, math, statistics
import torch.nn.functional as F
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not enabled')
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'


In [ ]:
#@title 2. Download upstream V4.3 and create a separately identified V4.4 variant
BASE_URL = 'https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/main/notebooks/Laya_V43_V31_Fidelity_First_Full_Then_Sliding_Colab.ipynb'
base_path=pathlib.Path('/content/Laya_V43_original_snapshot.ipynb')
urllib.request.urlretrieve(BASE_URL,base_path)
source=nbformat.read(base_path,as_version=4)
print('Upstream downloaded:',len(source.cells),'cells')

# Each strict replacement verifies the expected upstream code. If V4.3 changes,
# fail explicitly instead of silently producing an incorrect notebook.
def replace_once(cell_idx,old,new):
    cell=source.cells[cell_idx]
    assert cell.cell_type=='code',cell_idx
    if cell.source.count(old)!=1:
        raise RuntimeError(f'V4.3 changed: expected exactly one match in cell {cell_idx}: {old[:100]!r}')
    cell.source=cell.source.replace(old,new,1)

config_index=next(i for i,c in enumerate(source.cells) if c.cell_type=='code' and 'RUN_JOINT_RECOVERY=True' in c.source)
full_index=next(i for i,c in enumerate(source.cells) if c.cell_type=='code' and 'class PDeltaV28FastWindowMemory' in c.source)
print('Configuration cell',config_index,'architecture cell',full_index)

replace_once(config_index,'RUN_JOINT_RECOVERY=True','RUN_JOINT_RECOVERY=False')
replace_once(config_index,'REQUIRE_SPEEDUP_IN_GATED_MODE=False','REQUIRE_SPEEDUP_IN_GATED_MODE=True')
replace_once(config_index,'RESUME=True','RESUME=False')
replace_once(config_index,'BENCH_REPEATS=3','BENCH_REPEATS=8')
replace_once(config_index,'SELECTION_MODE="quality_gated"','SELECTION_MODE="quality_gated"')
replace_once(config_index,'OUT=Path("/content/Laya_V43_V31_Fidelity_First")',
             'OUT=Path("/content/Laya_V44_Fast_Fidelity")')
replace_once(config_index,'OUT=Path("/content/drive/MyDrive/TinyCeNN-LM/V43")',
             'OUT=Path("/content/drive/MyDrive/TinyCeNN-LM/V44")')
# V4.3 enforces 10 minimum training steps; keep original training contingency.
# The first skipped path is safe ONLY when downstream fidelity gates pass.
replace_once(config_index,'FULL_WINDOW=1024',
             'FAST_FULL_LOCAL_ONLY=True # experimental ablation; fidelity gates enforce fallback\nFULL_WINDOW=1024')
# This reduces FULL-branch extra computation; no weights are overwritten.
# For very short sequences the original path was already exact/near-native.
replace_once(full_index,'        landmarks = self._landmarks(q, k, v, valid)',
"""        # V4.4: fast branch ablation. Keep the original full attention path,
        # QKV, RoPE, value projection and FFN. Disable near-zero extras only.
        # Never deploy without PROBE speed/fidelity and untouched AUDIT.
        if FAST_FULL_LOCAL_ONLY:
            return self.finish(local * mask, hidden_states)
        landmarks = self._landmarks(q, k, v, valid)""")
# Avoid claiming old fingerprints and old V43 file names for changed architecture.
state_idx=next(i for i,c in enumerate(source.cells) if c.cell_type=='code' and 'experiment_version="v43-' in c.source)
replace_once(state_idx,'experiment_version="v43-v31-querykey-rope-a1"',
             'experiment_version="v44-fast-full-local-only-sdg-v1"')
replace_once(state_idx,'PROGRESS_PATH=OUT/"v43_progress.json"',
             'PROGRESS_PATH=OUT/"v44_progress.json"')
# Rename remaining generated artifact names in the notebook sources only.
for c in source.cells:
    if c.cell_type=='code':
        c.source=c.source.replace('"v43_', '"v44_')
source.cells.insert(0,nbformat.v4.new_markdown_cell(
"""# V4.4 — FAST FULL local-only ablation and strict CUDA speed gates

Generated from the original V4.3 upstream notebook. Only the adapter's
**additional** landmark/global/conv/routing branches are skipped in FULL layers
when `FAST_FULL_LOCAL_ONLY=True`. All original Q/K/V, RoPE, residuals, FFNs,
position and task evaluation code remain in place. Sliding operators are
unchanged in this first controlled variant. Speed gate is required.

**Caution:** This is an experimental approximation, not exact mathematical equivalence.
The original V4.3 model is not overwritten. Judge success by untouched AUDIT
and final end-to-end speed, not layerwise cosine alone."""))
source.metadata.setdefault('colab',{})['name']='Laya_V44_Fast_Fidelity_Experiment.ipynb'
out=pathlib.Path('/content/Laya_V44_Fast_Fidelity_Experiment.ipynb')
nbformat.write(source,out)
print('Created:',out,'cells:',len(source.cells))
print('Checks: strict speed=True; no joint recovery; isolated output namespace; ablation enabled')


In [ ]:
#@title 3. Verify modified notebook Python syntax and download
import ast
from IPython.display import display,FileLink
bad=[]
for i,c in enumerate(source.cells):
    if c.cell_type!='code':continue
    # IPython magics are syntactically valid in notebooks but not in ast.parse.
    code='\n'.join(ln for ln in c.source.splitlines() if not ln.lstrip().startswith(('%','!')))
    try:ast.parse(code)
    except SyntaxError as e:bad.append((i,str(e)))
assert not bad,f'Notebook syntax checks failed: {bad}'
print('Python AST syntax checks passed (not a runtime / GPU fidelity test).')
display(FileLink(str(out)))
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    pass


## Independent research experiment: true bounded-window SDPA

A dense attention mask is **not** a computationally sparse implementation. The function below computes only the key/value neighborhood needed by each query block. It uses standard PyTorch SDPA and is a *correctness-checked reference kernel*, not an optimized Triton implementation. On a T4, Python/block launch overhead can make it slower; let measurements decide. At query-block boundaries, the key range is expanded by `radius` on either side and an explicit per-query mask limits all attention to the selected neighborhood. It is bidirectional, matching ModernBERT local attention's semantics. Padding and RoPE must be handled by the caller; benchmark correctness here first with unpadded Q/K/V.

In [ ]:
#@title 4. Bounded-window reference implementation and exactness tests
@torch.no_grad()
def bounded_window_sdpa(q,k,v,radius,block_size=64):
    # q/k/v: [batch, heads, tokens, d], bidirectional symmetric window
    B,H,T,D=q.shape
    assert k.shape==v.shape==q.shape
    outputs=[]
    scale=D**-0.5
    for a in range(0,T,block_size):
        b=min(T,a+block_size)
        left=max(0,a-radius); right=min(T,b+radius)
        qi=torch.arange(a,b,device=q.device)[:,None]
        kj=torch.arange(left,right,device=q.device)[None,:]
        allowed=(kj>=qi-radius)&(kj<=qi+radius)
        x=F.scaled_dot_product_attention(q[:,:,a:b,:],k[:,:,left:right,:],
             v[:,:,left:right,:],attn_mask=allowed[None,None,:,:],dropout_p=0.0)
        outputs.append(x)
    return torch.cat(outputs,dim=2)

@torch.no_grad()
def dense_window_reference(q,k,v,radius):
    T=q.shape[2]
    p=torch.arange(T,device=q.device)
    allowed=(p[None,:]-p[:,None]).abs()<=radius
    return F.scaled_dot_product_attention(q,k,v,attn_mask=allowed[None,None],dropout_p=0.0)

DTYPE=torch.float16
for T in [33,127,257,512]:
    q=torch.randn(2,4,T,64,device='cuda',dtype=DTYPE)
    k=torch.randn_like(q);v=torch.randn_like(q)
    for r in [32,64,128]:
        x=bounded_window_sdpa(q,k,v,r,block_size=64)
        y=dense_window_reference(q,k,v,r)
        maxdiff=float((x-y).abs().max())
        cos=float(F.cosine_similarity(x.float().flatten(),y.float().flatten(),dim=0))
        print('T',T,'window',2*r+1,'max-abs',round(maxdiff,5),'cos',round(cos,7))
        assert torch.allclose(x.float(),y.float(),rtol=2e-2,atol=4e-3)
print('All synthetic GPU correctness tests passed.')


In [ ]:
#@title 5. CUDA-event paired kernel benchmarks on T4 (do not assume speedup)
import numpy as np
@torch.no_grad()
def ms_timed(f,warm=8,reps=20):
    for _ in range(warm):f()
    torch.cuda.synchronize()
    vals=[]
    for _ in range(reps):
        a=torch.cuda.Event(enable_timing=True);b=torch.cuda.Event(enable_timing=True)
        a.record();f();b.record();b.synchronize()
        vals.append(a.elapsed_time(b))
    return float(np.median(vals)),float(np.percentile(vals,95))

measurements=[]
for T in [128,256,512,1024]:
    q=torch.randn(2,8,T,64,device='cuda',dtype=DTYPE)
    k=torch.randn_like(q);v=torch.randn_like(q)
    for radius in [64,128,256]:
        base,pbase=ms_timed(lambda:dense_window_reference(q,k,v,radius))
        optimized,popt=ms_timed(lambda:bounded_window_sdpa(q,k,v,radius,64))
        measurements.append({'tokens':T,'window':2*radius+1,'dense_ms':base,
            'bounded_ms':optimized,'speedup':base/optimized,'dense_p95_ms':pbase,
            'bounded_p95_ms':popt})
        print(measurements[-1])
report_path=pathlib.Path('/content/v44_window_kernel_benchmark.json')
report_path.write_text(json.dumps({'gpu':torch.cuda.get_device_name(0),
        'pytorch':torch.__version__,'measurements':measurements},indent=2))
print('Saved',report_path)


## Next experiments needed for publishable novelty

1. Replace the Python block loop with a **Triton fused tiled sparse-attention kernel**, retaining accurate RoPE, valid-token masking and true bidirectional window semantics. Compare to the actual native Flash/SDPA backend, not only dense-masked SDPA.
2. Evaluate attention windows 129 / 257 / 512 on the **10 globally attending layers** individually, using PROBE to select and untouched AUDIT for final confirmation. A narrower window is a different operator; it is not parity by design.
3. Add **CeNN correction selectively** only for high-uncertainty tokens, and benchmark whether correction earns its own CUDA cost. Compare correction off / always on / uncertainty gated.
4. Report layer 15 individually, then search for a small jointly beneficial subset before replacing 28 layers. Benchmark short and long sequences separately, including batch size and peak VRAM. Keep the original native teacher immutable.

**Novelty hypothesis, not established claim:** sensitivity-aware, selectively gated CeNN correction coupled to true block-sparse attention can preserve task decisions while reducing arithmetic and memory traffic. Demonstrate it with real replacement kernels, decision fidelity, gold accuracy, latency and memory measurements. The reference implementation above is an experimental baseline, not yet a new fused kernel.